<a href="https://colab.research.google.com/github/elenamaggiore/Thesis_Elena_Maggiore/blob/preprocessing/PREPROCESSING_oversampling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PREPROCESSING: oversampling

This notebook will perform oversampling on the PANDORA test set, using a **GPT-2 model fine-tuned on the training chunks** (same approach used in the MBTI dataset notebook: label control codes + fine-tuning on the train set only + generation from real prompts).
The plan is broken down in the following steps:
* Setting up parameters
* Building the chunked dataset and the author-level train/validation split
* Creating the oversampling plan and verify its consistence
* Fine-tuning GPT-2 on the training chunks only and generating the synthetic chunks
* Check if the process was actually successful

Libraries:

In [1]:
!pip -q install transformers

import gc
import hashlib
import html
import json
import math
import random
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


Oversampling will be applied on the final dataset:

In [2]:
BASE = Path("/content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets_PANDORA")
INPUT_FILE = BASE / "final_complete_dataset.csv"
OUT_DIR = BASE / "Oversampling output" / "final"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_PATH = OUT_DIR / "author_split.csv"
FT_DIR = OUT_DIR / "gpt2_finetuned"   # fine-tuned checkpoint (re-used only if it was trained on the very same authors and settings)

AUTHOR_COL = "author"

General parameters:

In [3]:
# TASKS:
MBTI_TASKS = {"mbti_introverted": "introverted", "mbti_intuitive": "intuitive",
              "mbti_thinking": "thinking", "mbti_perceiving": "perceiving"}
B5_TASKS = {f"b5_{t}": t for t in ["openness", "conscientiousness", "extraversion",
                                   "agreeableness", "neuroticism"]}
TASKS = {**MBTI_TASKS, **B5_TASKS}

# BIG FIVE THRESHOLDS:
B5_THRESHOLDS = {
    "openness": 50,
    "conscientiousness": 50,
    "extraversion": 50,
    "agreeableness": 50,
    "neuroticism": 50,
}
B5_TIES_AS = None # What to do with record that are exactly on the threshold?

# CHUNKING PARAMETERS:
SEED = 11
CHUNK_WORDS = 150
MAX_CHUNKS_PER_AUTHOR = 10
MIN_COMMENT_WORDS = 10
MAX_COMMENT_WORDS = 200
MAX_RAW_SAMPLE = 200
VAL_SIZE = 0.30
CLEAN_COMMENTS = True   # unescape HTML entities, replace URLs with a placeholder, collapse whitespace inside each comment
URL_TOKEN = "URL"

# VERIFICATION:
LEAK_RE = re.compile(r"\b[ei][ns][tf][jp]s?\b|\b(?:[nsft][ei]){4}\b|TYPE_MENTION|COGFUNC_MENTION", re.I)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

Oversampling parameters:

In [4]:
TARGET_RATIO = 0.8           # expected ratio
MAX_VARIANTS = 2             # max synthetic texts generated from the same real chunk
OVERGENERATE = 1.2           # overgeneration to allow filtering
MAX_NEW_PER_TASK = 10_000

Model parameters (base model, fine-tuning, generation and quality filters):

In [5]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("WARNING: no GPU detected, fine-tuning and generation will be extremely slow. Change runtime to T4 GPU.")

GEN_MODEL = "gpt2"           # base model: it is FINE-TUNED on the training chunks before generating

# Fine-tuning (on the TRAIN chunks only, label-masked text)
FT_MAX_TOKENS = 384          # max length of a fine-tuning sequence (control codes + chunk); lower it if you run out of GPU memory
FT_EPOCHS = 2
FT_LR = 5e-5
FT_BATCH_SIZE = 16
FT_WARMUP_FRAC = 0.05
FT_MAX_EXAMPLES = None       # e.g. 15000 to cap the training time; None = all training chunks
FT_PPL_SAMPLE = 500          # validation chunks used only to monitor perplexity (never for training)
FT_LOG_EVERY = 50            # steps

# Generation
GEN_WINDOW_COMMENTS = 2      # comments of a real training chunk used as prompt (after the control codes)
PROMPT_MAX_TOKENS = 150      # context window size
GEN_TEMP, GEN_TOP_K, GEN_TOP_P = 0.7, 50, 0.95
NUM_BEAMS = 1
NO_REPEAT_NGRAM = 3          # chunks are longer than the posts of the MBTI notebook, so 2 would be too restrictive
GEN_LEN_RATIO = 1.0          # max new tokens / source chunk tokens
GEN_MIN_NEW, GEN_MAX_NEW = 80, 350
GEN_BATCH_SIZE = 32

# Quality filters
LEN_RATIO = (0.6, 1.5)       # generated words/source words
MAX_COPY_OVERLAP = 0.70      # max trigram overlap between source chunk and its synthetic text
MIN_DISTINCT_RATIO = 0.40    # min share of distinct words (discards looping/degenerate generations)

Other options are listed here:

In [6]:
# GEN_MODEL = "gpt2-medium"   # 355M parameters: better text, ~3x slower; use FT_BATCH_SIZE = 8 on a T4

Functions for extracting comments, chunking them, and building the dataset:

In [7]:
RE_URL = re.compile(r"(?:https?://|www\.)\S+", re.I)

# Control codes: one special token per task and value (e.g. <|b5_openness_1|>). Unlike the 16 type codes of the MBTI notebook,
# these tokens are seen in many chunks each, which matters with only a few hundred authors.
CTRL_TOKENS = [f"<|{t}_{v}|>" for t in TASKS for v in (0, 1)]


def clean_comment(c): # Cleans a single comment
    """Unescapes HTML, replaces URLs with a placeholder and collapses whitespace (so '\n' only separates comments)."""
    c = str(c)
    if not CLEAN_COMMENTS:
        return c.strip()
    c = html.unescape(c)
    c = RE_URL.sub(f" {URL_TOKEN} ", c)
    return re.sub(r"\s+", " ", c).strip()


def label_cols(): # Only considers the columns related to the task, if you're running the code for a specific task
    cols = list(TASKS.values()) + ["mbti"]
    return list(dict.fromkeys(cols))


def load_authors(file): # Reads through the dataset and collects comments per author
    """Reads CSV and returns authors, labels, comments, and stats."""
    cols = label_cols()
    wanted = set(cols) | {AUTHOR_COL, "comments"}
    labels, cands, stats = {}, {}, Counter()

    for part in pd.read_csv(file, usecols=lambda c: c in wanted, chunksize=100, low_memory=False):
        missing = (set(TASKS.values()) | {AUTHOR_COL, "comments"}) - set(part.columns)
        if missing:
            raise KeyError(f"Colonne mancanti nel file: {sorted(missing)}")
        for row in part.to_dict("records"):
            aid = str(row[AUTHOR_COL])
            raw = row.get("comments")
            comments = json.loads(raw) if isinstance(raw, str) else []
            stats["author_rows"] += 1
            stats["comments_in"] += len(comments)

            good = []
            for c in comments:
                c = clean_comment(c)
                n = len(c.split())
                if n < MIN_COMMENT_WORDS:
                    stats["too_short"] += 1
                elif n > MAX_COMMENT_WORDS:
                    stats["too_long"] += 1
                else:
                    good.append(c)
            if len(good) > MAX_RAW_SAMPLE:
                good = random.Random(f"{SEED}-{aid}").sample(good, MAX_RAW_SAMPLE)
            stats["comments_kept"] += len(good)

            cands.setdefault(aid, []).extend(good)
            labels.setdefault(aid, {c: row.get(c) for c in cols})
    gc.collect()
    auth = pd.DataFrame.from_dict(labels, orient="index").rename_axis("author_id").reset_index()
    return auth, cands, stats


def _chunk_row(aid, k, cur, n): # Creates a dictionary with all chunks
    return dict(chunk_id=f"{aid}_{k}", author_id=aid, text="\n".join(cur),
                comments_json=json.dumps(cur, ensure_ascii=False), n_words=n, n_comments=len(cur))


def build_chunks(cands): # Builds chunks
    rows, dropped = [], 0
    for aid, comments in cands.items():
        comments = list(dict.fromkeys(comments))
        random.Random(f"{SEED}-{aid}").shuffle(comments)
        cur, n, k = [], 0, 0
        for c in comments:
            cur.append(c)
            n += len(c.split())
            if n >= CHUNK_WORDS:
                rows.append(_chunk_row(aid, k, cur, n))
                k, cur, n = k + 1, [], 0
                if k >= MAX_CHUNKS_PER_AUTHOR:
                    break
        if k == 0 and n >= CHUNK_WORDS // 2:
            rows.append(_chunk_row(aid, 0, cur, n))
            k = 1
        dropped += int(k == 0)
    return pd.DataFrame(rows), dropped


def add_task_labels(df):
    """Creates taks columns (0/1/NaN)."""
    for task, col in MBTI_TASKS.items():
        s = pd.to_numeric(df[col], errors="coerce")
        bad = s.notna() & ~s.isin([0, 1])
        if bad.any():
            print(f"WARNING: {col} has {bad.sum()} non binary values -> NaN")
            s = s.mask(bad)
        df["y_" + task] = s.astype("float64")

    # Big Five thresholding
    for task, col in B5_TASKS.items():
        thr = B5_THRESHOLDS[col]
        s = pd.to_numeric(df[col], errors="coerce")
        y = pd.Series(np.where(s > thr, 1.0, np.where(s < thr, 0.0, np.nan)), index=s.index)
        if B5_TIES_AS is not None:
            y[s == thr] = float(B5_TIES_AS)
        df["y_" + task] = y.astype("float64")
    return df


def ctrl_code(row): # Control code of a chunk: one token for each defined task label
    return "".join(f"<|{t}_{int(row['y_' + t])}|>" for t in TASKS if pd.notna(row["y_" + t]))


def assign_split(chunks, path): # Splits each authors in training and validation
    """Splits each author in training and validation, saves on file."""
    if Path(path).exists():
        sp = pd.read_csv(path, dtype={"author_id": str})
    else:
        a = chunks.drop_duplicates("author_id")
        strat = None
        if "mbti" in a and a["mbti"].notna().all() and a["mbti"].value_counts().min() >= 2:
            strat = a["mbti"]
        tr, va = train_test_split(a["author_id"], test_size=VAL_SIZE, stratify=strat, random_state=SEED)
        sp = pd.concat([pd.DataFrame({"author_id": tr, "split": "train"}),
                        pd.DataFrame({"author_id": va, "split": "val"})], ignore_index=True)
        sp.to_csv(path, index=False)
        print("Split saved in", path, "| stratified on:", strat is not None)
    out = chunks.merge(sp, on="author_id", how="left")
    if out["split"].isna().any():
        print(f"WARNING: {out['split'].isna().sum()} chunk removed because author was not in split")
        out = out[out["split"].notna()]
    return out.reset_index(drop=True)


def build_dataset(file=INPUT_FILE, split_path=SPLIT_PATH): # Builds the dataset that will be used for oversampling
    auth, cands, stats = load_authors(file)
    print("Comments:", dict(stats))
    chunks, dropped = build_chunks(cands)
    print(f"Authors without chunks are discarded: {dropped:,}")
    chunks = add_task_labels(chunks.merge(auth, on="author_id", how="left"))
    chunks["ctrl"] = chunks.apply(ctrl_code, axis=1)
    return assign_split(chunks, split_path)

Functions for creating the oversampling plan and verify its consistence:

In [8]:
def generation_plan(train): # Calculates the number of samples to generate, for each task
    """For each task, labels the minority class, counts how many samples are need and how many to generate."""
    rows = []
    for task in TASKS:
        y = train["y_" + task].dropna()
        vc = y.value_counts()
        if len(vc) < 2:
            rows.append(dict(task=task, n_pos=int(vc.get(1.0, 0)), n_neg=int(vc.get(0.0, 0)),
                             minority=np.nan, ratio=np.nan, n_needed=0, n_to_generate=0))
            continue
        minority, n_min, n_maj = vc.idxmin(), vc.min(), vc.max()
        needed = int(max(0, min(TARGET_RATIO * n_maj - n_min, MAX_NEW_PER_TASK)))
        rows.append(dict(task=task, n_pos=int(vc.get(1.0, 0)), n_neg=int(vc.get(0.0, 0)),
                         minority=minority, ratio=round(n_min / n_maj, 3),
                         n_needed=needed, n_to_generate=int(np.ceil(needed * OVERGENERATE))))
    return pd.DataFrame(rows)


def _ratio(s): # Calculates the ratio
    vc = s.value_counts()
    return vc.min() / vc.max() if len(vc) > 1 else np.nan


def _shingles(text, n=8): # Hashed word n-grams, used for the memorisation check
    w = text.lower().split()
    return {hash(" ".join(w[i:i + n])) for i in range(max(0, len(w) - n + 1))}


def verify_oversampling(train, synth, val, tasks=None): # Checks if the oversampling was successful
    """
    train, val : Real dataframe
    synth      : Generated dataframe
                 chunk_id, src_chunk_id, author_id, task, text, y_*, len_ratio, overlap, distinct, variant
    """
    tasks = list(tasks or TASKS)
    ycols = [f"y_{t}" for t in tasks]

    # 1. Split integrity
    assert not set(train.author_id) & set(val.author_id), "Authors in common between train and validation"
    assert synth.author_id.isin(train.author_id).all(), "Comments generated from authors not in train"
    assert synth.src_chunk_id.isin(train.chunk_id).all(), "Sources not in train"
    print("1. Okay: no leakage detected")

    # 2. Validation balance
    rows = []
    for t in tasks:
        y = f"y_{t}"
        tr, sy = train[y].dropna(), synth.loc[synth.task == t, y].dropna()
        rows.append(dict(task=t, n_real=len(tr), n_synth=len(sy), ratio_before=_ratio(tr),
                         ratio_after=_ratio(pd.concat([tr, sy])), ratio_val=_ratio(val[y].dropna())))
    print("2. Balance:\n", pd.DataFrame(rows).round(3).to_string(index=False))

    # 3. Quality check
    print("3. Metrics:\n", synth[["len_ratio", "overlap", "distinct"]].describe().round(3).to_string())
    assert synth.len_ratio.between(*LEN_RATIO).all()
    assert (synth.overlap <= MAX_COPY_OVERLAP).all()
    assert (synth.distinct >= MIN_DISTINCT_RATIO).all()
    assert synth.groupby(["src_chunk_id", "task"]).size().max() <= MAX_VARIANTS
    print("Quality check approved")

    # 4. Label leakage check
    n_leak = synth.text.str.contains(LEAK_RE).sum()
    print(f"4. Generated: {n_leak} comments with label leakage")

    # 5. Changes in label correlation
    d = (pd.concat([train, synth])[ycols].corr() - train[ycols].corr()).abs().max().max()
    print(f"5. Maximum change in label correlation: {d:.3f}")

    # 6. Memorisation check: 8-word shingles shared with ANY real training chunk
    uniq = synth.drop_duplicates(["src_chunk_id", "variant"])
    train_sh = set()
    for t in train.text:
        train_sh |= _shingles(t)
    ov = []
    for t in uniq.text.sample(min(len(uniq), 1000), random_state=SEED):
        sh = _shingles(t)
        ov.append(len(sh & train_sh) / len(sh) if sh else 0.0)
    ov = np.array(ov)
    print(f"6. Memorisation: exact duplicates {uniq.text.duplicated().mean() * 100:.1f}% | "
          f"mean shingle overlap {ov.mean():.3f} | texts with overlap > 50%: {(ov > 0.5).mean() * 100:.1f}%")
    del train_sh; gc.collect()

    # 7. AUC checks
    n = min(len(train), len(synth))
    X = pd.concat([train.text.sample(n, random_state=SEED), synth.text.sample(n, random_state=SEED)])
    y = [0] * n + [1] * n
    auc = cross_val_score(make_pipeline(TfidfVectorizer(max_features=20000, sublinear_tf=True),
                                        LogisticRegression(max_iter=1000)),
                          X, y, cv=5, scoring="roc_auc").mean()
    print(f"7. Real AUC vs generated AUC: {auc:.3f} (1 means that they can be distinguished)")

    # 8. F1 without oversampling
    def fit_eval(tr_df, y, **kw):
        m = make_pipeline(TfidfVectorizer(max_features=20000, sublinear_tf=True, ngram_range=(1, 2)),
                          LogisticRegression(max_iter=1000, **kw))
        tr_df = tr_df.dropna(subset=[y])
        m.fit(tr_df.text, tr_df[y])
        v = val.dropna(subset=[y])
        return f1_score(v[y], m.predict(v.text), average="macro")

    res = []
    for t in tasks:
        y = f"y_{t}"
        tr = train.dropna(subset=[y])
        vc = tr[y].value_counts()
        minority = vc.idxmin()
        n_extra = int(max(0, TARGET_RATIO * vc.max() - vc.min()))
        rand = pd.concat([tr, tr[tr[y] == minority].sample(n_extra, replace=True, random_state=SEED)])
        res.append(dict(task=t,
                        base=fit_eval(tr, y),
                        class_weight=fit_eval(tr, y, class_weight="balanced"),
                        random_os=fit_eval(rand, y),
                        llm_os=fit_eval(pd.concat([tr, synth[synth.task == t]]), y)))
    print("8. Macro-F1 on validation (llm_os = fine-tuned GPT-2 oversampling):\n", pd.DataFrame(res).round(3).to_string(index=False))

Configurations: selecting "train" only oversamples on train set and uses validation set to assess quality, "None" oversamples on the whole dataset.

In [9]:
SOURCE_SPLIT = "train"

MAX_ROUNDS = 3                 # generation and filter rounds
GEN_TAG = "gpt2_ft_v1"         # model

SYNTH_PATH = OUT_DIR / "synthetic_chunks.csv"
REJECT_PATH = OUT_DIR / "rejected_synthetic.csv"
REPORT_PATH = OUT_DIR / "oversampling_report.csv"

CJK_RE = re.compile(r"[\u3040-\u30ff\u3400-\u9fff\uac00-\ud7af]")

Tokenizer with control codes, prompt building and text utilities:

In [10]:
def _md5(s): # Hash conversion
    return hashlib.md5(s.encode("utf-8")).hexdigest()


def _norm(t): # Normalisation
    return re.sub(r"\W+", " ", t.lower()).strip()


def _trigrams(t): # Trigrams
    w = re.findall(r"[a-z0-9']+", t.lower())
    return {tuple(w[i:i + 3]) for i in range(len(w) - 2)}


def get_tokenizer(path=GEN_MODEL): # Tokenizer with the control codes as special tokens
    tok = AutoTokenizer.from_pretrained(path)
    tok.add_special_tokens({"additional_special_tokens": CTRL_TOKENS})   # no-op if they are already there
    tok.pad_token = tok.eos_token
    tok.padding_side = "left"                      # left for generation (switched to right only while fine-tuning)
    return tok


def build_prompt(code, comments, cid, variant): # Control code + a window of real comments of the source chunk
    rng = random.Random(f"{SEED}-{cid}-{variant}")
    s = rng.randrange(0, max(1, len(comments) - GEN_WINDOW_COMMENTS + 1))
    return f"{code} " + "\n".join(comments[s:s + GEN_WINDOW_COMMENTS])

Fine-tuning GPT-2 on the training chunks (train authors only, chunks with label mentions excluded):

In [11]:
def _encode_train(tok, texts):
    tok.padding_side = "right"                       # right padding for training
    enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=FT_MAX_TOKENS)
    tok.padding_side = "left"
    labels = enc["input_ids"].clone()
    labels[enc["attention_mask"] == 0] = -100        # padding is ignored by the loss (the real EOS is kept)
    enc["labels"] = labels
    return {k: v.to(DEVICE) for k, v in enc.items()}


@torch.no_grad()
def perplexity(model, tok, texts, bs=32):
    model.eval()
    tot_loss, tot_tok = 0.0, 0
    for i in range(0, len(texts), bs):
        enc = _encode_train(tok, texts[i:i + bs])
        n = int((enc["labels"][:, 1:] != -100).sum())
        tot_loss += model(**enc).loss.item() * n
        tot_tok += n
    return float(np.exp(tot_loss / tot_tok))


def finetune_gpt2(tok, train_texts, val_texts):
    model = AutoModelForCausalLM.from_pretrained(GEN_MODEL)
    model.resize_token_embeddings(len(tok))          # make room for the control codes
    model.to(DEVICE)
    if val_texts:
        print(f"Validation perplexity BEFORE fine-tuning: {perplexity(model, tok, val_texts):.1f}")
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=0.01)
    n_steps = FT_EPOCHS * int(np.ceil(len(train_texts) / FT_BATCH_SIZE))
    sched = get_linear_schedule_with_warmup(opt, int(FT_WARMUP_FRAC * n_steps), n_steps)
    use_amp = DEVICE == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    rng = np.random.default_rng(SEED)
    step, run, t0 = 0, 0.0, time.time()
    for ep in range(FT_EPOCHS):
        order = rng.permutation(len(train_texts))
        for i in range(0, len(order), FT_BATCH_SIZE):
            enc = _encode_train(tok, [train_texts[j] for j in order[i:i + FT_BATCH_SIZE]])
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                loss = model(**enc).loss
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            opt.zero_grad(set_to_none=True); sched.step()
            step += 1; run += loss.item()
            if step % FT_LOG_EVERY == 0:
                print(f"  epoch {ep + 1}/{FT_EPOCHS} | step {step}/{n_steps} | loss {run / FT_LOG_EVERY:.3f} | {(time.time() - t0) / 60:.1f} min")
                run = 0.0
        if val_texts:
            print(f"Validation perplexity after epoch {ep + 1}: {perplexity(model, tok, val_texts):.1f}")
        model.train()
    del opt, sched, scaler
    gc.collect(); torch.cuda.empty_cache()
    return model.eval()


def ft_signature(authors): # Identifies the training authors and settings a checkpoint was trained with
    cfg = json.dumps([GEN_MODEL, FT_MAX_TOKENS, FT_EPOCHS, FT_LR, FT_BATCH_SIZE, FT_WARMUP_FRAC, FT_MAX_EXAMPLES,
                      CHUNK_WORDS, MAX_CHUNKS_PER_AUTHOR, MIN_COMMENT_WORDS, MAX_COMMENT_WORDS, CLEAN_COMMENTS,
                      SEED, CTRL_TOKENS, sorted(map(str, authors))])
    return _md5(cfg)


def load_generator(train, val, sig): # Loads the fine-tuned checkpoint if it was trained on these same authors, otherwise fine-tunes GPT-2
    assert val is None or not (set(train.author_id) & set(val.author_id)), "Authors in common between train and validation"
    meta_file = FT_DIR / "ft_meta.json"
    if (FT_DIR / "config.json").exists() and meta_file.exists() and json.loads(meta_file.read_text()).get("signature") == sig:
        print("Loading the fine-tuned model from", FT_DIR)
        tok = get_tokenizer(FT_DIR)
        model = AutoModelForCausalLM.from_pretrained(FT_DIR)
    else:
        if (FT_DIR / "config.json").exists():
            print("WARNING: the checkpoint in FT_DIR was trained on different authors/settings: it is ignored and overwritten.")
        tok = get_tokenizer()
        src = train if FT_MAX_EXAMPLES is None else train.sample(min(FT_MAX_EXAMPLES, len(train)), random_state=SEED)
        ft_texts = [f"{c} {t}{tok.eos_token}" for c, t in zip(src["ctrl"], src["text"])]
        val_texts = []
        if val is not None and len(val):
            v = val.sample(min(FT_PPL_SAMPLE, len(val)), random_state=SEED)
            val_texts = [f"{c} {t}{tok.eos_token}" for c, t in zip(v["ctrl"], v["text"])]
        print(f"Fine-tuning on {len(ft_texts):,} training chunks ({FT_EPOCHS} epochs)")
        model = finetune_gpt2(tok, ft_texts, val_texts)
        model.save_pretrained(FT_DIR); tok.save_pretrained(FT_DIR)
        meta_file.write_text(json.dumps({"signature": sig, "model": GEN_MODEL, "n_train_chunks": len(ft_texts),
                                         "n_train_authors": int(train.author_id.nunique())}))
    if DEVICE == "cuda":
        model = model.half()
    model = model.to(DEVICE).eval()
    gc.collect(); torch.cuda.empty_cache()
    return tok, model

Generation, quality filters and the whole oversampling procedure:

In [12]:
# Generation:
def raw_cache_path(sig): # Cache file: it changes with the model, the training authors and the generation settings
    cfg = json.dumps([sig, GEN_MODEL, GEN_TAG, GEN_WINDOW_COMMENTS, PROMPT_MAX_TOKENS, GEN_TEMP, GEN_TOP_K, GEN_TOP_P,
                      NUM_BEAMS, NO_REPEAT_NGRAM, GEN_LEN_RATIO, GEN_MIN_NEW, GEN_MAX_NEW])
    return OUT_DIR / f"raw_generations_{_md5(cfg)[:8]}.jsonl"


def load_cache(src_text, path):
    print("Cache file:", path.name)
    cache = {}
    if path.exists():
        with open(path, encoding="utf-8") as fh:
            for line in fh:
                try:
                    r = json.loads(line)
                except json.JSONDecodeError:
                    continue
                s = r["src"]
                if s in src_text and r["md5"] == _md5(src_text[s]):
                    cache[(s, r["v"])] = r
    print(f"Cache: {len(cache):,} available generated texts.")
    return cache


@torch.no_grad()
def generate_batch(tok, model, prompts, max_new):
    """Takes a batch and generates the continuation (the prompt is NOT returned). If OOM, splits the batch and tries again."""
    try:
        enc = tok(prompts, return_tensors="pt", padding=True, truncation=True, max_length=PROMPT_MAX_TOKENS).to(model.device)
        out = model.generate(**enc, do_sample=True, top_k=GEN_TOP_K, top_p=GEN_TOP_P, temperature=GEN_TEMP,
                             num_beams=NUM_BEAMS, no_repeat_ngram_size=NO_REPEAT_NGRAM, max_new_tokens=max_new,
                             pad_token_id=tok.eos_token_id)
    except torch.cuda.OutOfMemoryError:
        if len(prompts) == 1:
            raise
        gc.collect()
        torch.cuda.empty_cache()
        h = len(prompts) // 2
        return generate_batch(tok, model, prompts[:h], max_new) + generate_batch(tok, model, prompts[h:], max_new)
    return tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)


def generate_jobs(jobs, src, tok, model, cache, path): # Creates jobs
    torch.manual_seed(SEED)
    jobs = sorted(jobs, key=lambda j: len(src[j[0]]["text"]))     # puts texts of similar length in the same batch
    with open(path, "a", encoding="utf-8") as fh:
        for i in tqdm(range(0, len(jobs), GEN_BATCH_SIZE), desc="GPT-2"):
            batch = jobs[i:i + GEN_BATCH_SIZE]
            prompts = [build_prompt(src[s]["ctrl"], src[s]["comments"], s, v) for s, v in batch]
            max_src = max(len(tok(src[s]["text"]).input_ids) for s, _ in batch)
            max_new = int(np.clip(GEN_LEN_RATIO * max_src, GEN_MIN_NEW, GEN_MAX_NEW))
            for (s, v), text in zip(batch, generate_batch(tok, model, prompts, max_new)):
                rec = dict(src=s, v=v, md5=_md5(src[s]["text"]), text=text.strip())
                cache[(s, v)] = rec
                fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
            fh.flush()


def qualitative_check(tok, model, train): # Texts generated from the control codes alone
    print("Samples generated from the control code alone:")
    for code_ in train["ctrl"].drop_duplicates().head(2):
        print(f"[{code_}]")
        for g in generate_batch(tok, model, [code_] * 2, 100):
            print("  -", g.strip()[:250].replace("\n", " / "))


# Filters:
def new_state(pool): # Sets new state
    return dict(judged=set(), acc=[], rej=[], seen=set(pool.text.map(_norm)), var_tris=defaultdict(list))


def judge_new(cache, state, src): # Judges each generated text
    """Judges new generated batches and calculates metrics."""
    new = sorted(k for k in cache if k not in state["judged"])
    for k in new:
        state["judged"].add(k)
        t, s_txt = cache[k]["text"], src[k[0]]["text"]
        ws, wt = len(s_txt.split()), len(t.split())
        len_ratio = wt / ws if ws else np.nan
        words = t.lower().split()
        distinct = len(set(words)) / len(words) if words else 0.0
        tri_s, tri_t = _trigrams(s_txt), _trigrams(t)
        overlap = len(tri_s & tri_t) / len(tri_s) if tri_s else 0.0
        nt = _norm(t)

        if wt < 5:
            reason = "empty"
        elif CJK_RE.search(t):
            reason = "non_english"
        elif LEAK_RE.search(t):
            reason = "leak"
        elif not (LEN_RATIO[0] <= len_ratio <= LEN_RATIO[1]):
            reason = "length"
        elif distinct < MIN_DISTINCT_RATIO:
            reason = "degenerate"                   # looping / low-diversity text
        elif overlap > MAX_COPY_OVERLAP:
            reason = "copy"
        elif nt in state["seen"]:
            reason = "duplicate"
        elif any(len(tri_t & o) / max(1, len(tri_t)) > MAX_COPY_OVERLAP for o in state["var_tris"][k[0]]):
            reason = "variant_too_similar"          # The variant is too similar to another unit
        else:
            reason = None

        rec = dict(src_chunk_id=k[0], variant=k[1], text=t, len_ratio=len_ratio, overlap=overlap, distinct=distinct)
        if reason is None:
            state["seen"].add(nt)
            state["var_tris"][k[0]].append(tri_t)
            rec["rand"] = random.Random(f"{SEED}-{k[0]}-{k[1]}").random()
            state["acc"].append(rec)
        else:
            rec["reason"] = reason
            state["rej"].append(rec)


def accepted_df(state): # Dataframe for accepted generations only
    cols = ["src_chunk_id", "variant", "text", "len_ratio", "overlap", "distinct", "rand"]
    return pd.DataFrame(state["acc"], columns=cols)


# Job Planning:
def candidate_pairs(minority_ids, cache, rng): # Counts pairs (chunk, variant) to test
    """Couples (chunk, variant) to try:"""
    pairs = []
    for v in range(MAX_VARIANTS):
        ids = [s for s in minority_ids if (s, v) not in cache]
        rng.shuffle(ids)
        pairs += [(s, v) for s in ids]
    return pairs


def run_oversampling(pool, val=None): # Runs the whole procedure
    """pool: real chunks (train). val: only used to monitor perplexity. Returns dataframe of generated chunks."""
    pool = pool.reset_index(drop=True)
    plan = generation_plan(pool).set_index("task")
    print("Oversampling plan:\n", plan.to_string())

    # Removes leaky chunks: they are neither prompts nor fine-tuning data (the model never sees label mentions)
    leaky = pool.text.str.contains(LEAK_RE)
    print(f"Chunks with label leaking: {int(leaky.sum())}")
    src_pool = pool[~leaky].reset_index(drop=True)
    src = {r.chunk_id: dict(text=r.text, comments=json.loads(r.comments_json), ctrl=r.ctrl) for r in src_pool.itertuples()}
    src_text = {k: v["text"] for k, v in src.items()}

    minority_ids, n_needed = {}, {}
    for task in TASKS:
        row = plan.loc[task]
        if row.n_needed <= 0 or pd.isna(row.minority):
            continue                                         # task is balanced
        ids = src_pool.loc[src_pool["y_" + task] == row.minority, "chunk_id"].tolist()
        minority_ids[task], n_needed[task] = ids, int(row.n_needed)
        if n_needed[task] > len(ids) * MAX_VARIANTS:
            print(f"WARNING {task}: {n_needed[task]} generations needed but in {len(ids)} chunks "
                  f"and MAX_VARIANTS={MAX_VARIANTS} the maximum number of generations is {len(ids) * MAX_VARIANTS}")

    sig = ft_signature(src_pool.author_id.unique())
    path = raw_cache_path(sig)
    cache = load_cache(src_text, path)
    state = new_state(pool)
    rng = random.Random(SEED)
    tok = model = None

    for rnd in range(1, MAX_ROUNDS + 1):
        judge_new(cache, state, src)
        acc = accepted_df(state)
        short = {t: max(0, n_needed[t] - int(acc.src_chunk_id.isin(minority_ids[t]).sum())) for t in n_needed}
        print(f"\nRound {rnd} | is missing for task: { {t: s for t, s in short.items() if s > 0} or 'none' }")

        jobs = set()
        for task, sh in short.items():
            if sh > 0:
                jobs.update(candidate_pairs(minority_ids[task], cache, rng)[:math.ceil(sh * OVERGENERATE)])
        if not jobs:
            break
        if tok is None:
            tok, model = load_generator(src_pool, val, sig)      # fine-tunes (or loads) GPT-2 only when needed
            qualitative_check(tok, model, src_pool)
        print(f"Needed : {len(jobs):,} generations.")
        generate_jobs(sorted(jobs), src, tok, model, cache, path)
    judge_new(cache, state, src)

    if model is not None:                                    # free GPU
        del model, tok
        gc.collect()
        torch.cuda.empty_cache()

    acc = accepted_df(state)
    src_idx = pool.set_index("chunk_id")
    ycols = [c for c in pool.columns if c.startswith("y_")]
    rows = []
    for task, ids in minority_ids.items():
        cand = acc[acc.src_chunk_id.isin(ids)].sort_values(["variant", "rand"]).head(n_needed[task])
        for r in cand.itertuples():
            base = src_idx.loc[r.src_chunk_id]
            d = dict(chunk_id=f"syn_{r.src_chunk_id}_v{r.variant}_{task}", src_chunk_id=r.src_chunk_id,
                     author_id=base["author_id"], task=task, text=r.text)
            d.update({c: base[c] for c in ycols})
            d.update(len_ratio=r.len_ratio, overlap=r.overlap, distinct=r.distinct, variant=r.variant)
            rows.append(d)
    out_cols = ["chunk_id", "src_chunk_id", "author_id", "task", "text"] + ycols + \
               ["len_ratio", "overlap", "distinct", "variant"]
    synth = pd.DataFrame(rows, columns=out_cols)

    # Results:
    rep = []
    for task in TASKS:
        y = pool["y_" + task].dropna()
        vc = y.value_counts()
        n_s = int((synth.task == task).sum())
        n_min = int(vc.min()) if len(vc) > 1 else 0
        n_maj = int(vc.max()) if len(vc) > 1 else 0
        rep.append(dict(task=task, n_min=n_min, n_maj=n_maj, n_needed=n_needed.get(task, 0), n_synth=n_s,
                        ratio_before=round(n_min / n_maj, 3) if n_maj else np.nan,
                        ratio_after=round((n_min + n_s) / n_maj, 3) if n_maj else np.nan))
    rep = pd.DataFrame(rep)
    rej = pd.DataFrame(state["rej"])
    print("\nOversampling results:\n", rep.to_string(index=False))
    if len(rej):
        print("\nDiscarded:\n", rej.reason.value_counts().to_string())
        print(f"Accepting ratio: {len(acc) / (len(acc) + len(rej)):.1%}")
        rej.to_csv(REJECT_PATH, index=False)
    rep.to_csv(REPORT_PATH, index=False)
    synth.to_csv(SYNTH_PATH, index=False)
    print(f"\nSaved: {SYNTH_PATH.name} ({len(synth):,} rows), {REPORT_PATH.name}, {REJECT_PATH.name}")
    return synth


def show_examples(synth, pool, n=3): # Prints some oversampled couples
    """Manually evaluation to add human control:"""
    if synth.empty:
        return
    src = pool.set_index("chunk_id").text
    for r in synth.sample(min(n, len(synth)), random_state=SEED).itertuples():
        print("=" * 80, f"\n[{r.task}] len_ratio={r.len_ratio:.2f} overlap={r.overlap:.2f} distinct={r.distinct:.2f}")
        print("SOURCE (the prompt is a window of this chunk)\n" + src[r.src_chunk_id][:900])
        print("GENERATED\n" + r.text[:900])

Execution:

In [13]:
if __name__ == "__main__":
    data = build_dataset()
    if SOURCE_SPLIT is None:
        print("WARNING: no validation will be performed. Please ensure you're not evaluating a model on the same dataset you've trained it. ")
        pool, val = data, None
    else:
        pool = data[data.split == SOURCE_SPLIT].reset_index(drop=True)
        val = data[data.split != SOURCE_SPLIT].reset_index(drop=True)

    synth = run_oversampling(pool, val)
    show_examples(synth, pool)

    if val is not None and len(synth):
        verify_oversampling(pool, synth, val)
    else:
        print("No validation was performed.")

Comments: {'author_rows': 377, 'comments_in': 895742, 'too_short': 166568, 'too_long': 32247, 'comments_kept': 60403}
Authors without chunks are discarded: 3
Oversampling plan:
                       n_pos  n_neg  minority  ratio  n_needed  n_to_generate
task                                                                        
mbti_introverted       1762    700       0.0  0.397       709            851
mbti_intuitive         2170    292       0.0  0.135      1444           1733
mbti_thinking          1521    941       0.0  0.619       275            330
mbti_perceiving        1376   1086       0.0  0.789        14             17
b5_openness            1714    616       0.0  0.359       755            906
b5_conscientiousness    928   1534       1.0  0.605       299            359
b5_extraversion         720   1692       1.0  0.426       633            760
b5_agreeableness        897   1465       1.0  0.612       275            330
b5_neuroticism         1036   1376       1.0  0.753 

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Samples generated from the control code alone:
[<|mbti_introverted_1|><|mbti_intuitive_1|><|mbti_thinking_0|><|mbti_perceiving_1|><|b5_openness_1|><|b5_conscientiousness_0|><|b5_extraversion_0|><|b5_agreeableness_1|><|b5_neuroticism_0|>]
  - I'm pretty sure I've had enough of this. / I don't know about you, but I'm not that into being around people who don't like me because they feel like I'm trying to be nice. I don't want to be rude, I want to get a little bit more comfortable in my own 
  - I'm gonna assume you're pretty good at it. I am. I'm not that great at it, though. / This is not an exaggeration. I've been a fan since the 90s. I used to be a teenager and I remember how much I loved the music. I loved it when I saw it on MTV. I did n
[<|mbti_introverted_0|><|mbti_intuitive_0|><|mbti_thinking_1|><|mbti_perceiving_1|><|b5_openness_0|><|b5_conscientiousness_1|><|b5_extraversion_1|><|b5_agreeableness_0|><|b5_neuroticism_0|>]
  - It's not just about the numbers, though. It's about t

GPT-2:   0%|          | 0/65 [00:00<?, ?it/s]


Round 2 | is missing for task: {'mbti_introverted': 305, 'mbti_intuitive': 1267, 'b5_openness': 369, 'b5_extraversion': 218}
Needed : 31 generations.


GPT-2:   0%|          | 0/1 [00:00<?, ?it/s]


Round 3 | is missing for task: {'mbti_introverted': 303, 'mbti_intuitive': 1267, 'b5_openness': 369, 'b5_extraversion': 206}

Oversampling results:
                 task  n_min  n_maj  n_needed  n_synth  ratio_before  ratio_after
    mbti_introverted    700   1762       709      406         0.397        0.628
      mbti_intuitive    292   2170      1444      177         0.135        0.216
       mbti_thinking    941   1521       275      275         0.619        0.799
     mbti_perceiving   1086   1376        14       14         0.789        0.799
         b5_openness    616   1714       755      386         0.359        0.585
b5_conscientiousness    928   1534       299      299         0.605        0.800
     b5_extraversion    720   1692       633      427         0.426        0.678
    b5_agreeableness    897   1465       275      275         0.612        0.800
      b5_neuroticism   1036   1376        64       64         0.753        0.799

Discarded:
 reason
length    1059
empty

Preparing the file for the GitHub commit:

In [14]:
!pip -q install nbstripout

SRC = "/content/drive/MyDrive/Thesis_Elena_Maggiore/Code/PREPROCESSING: oversampling.ipynb"
DST = "/content/drive/MyDrive/Thesis_Elena_Maggiore/Code/GIT_READY/PREPROCESSING_oversampling.ipynb"

!mkdir -p "/content/drive/MyDrive/Thesis_Elena_Maggiore/Code/GIT_READY"
!cp "{SRC}" "{DST}"
!nbstripout --extra-keys "metadata.colab cell.metadata.colab cell.metadata.executionInfo cell.metadata.outputId" "{DST}"
!grep -c "displayName\|userId" "{DST}"
!grep -c '"outputs": \[\]' "{DST}"

1
13
